<a href="https://colab.research.google.com/github/dhairya-p/bc3412-code/blob/main/bc3412_analysis_try.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [126]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.discrete.discrete_model import Logit
import statsmodels.formula.api as smf
import statsmodels.genmod.generalized_estimating_equations as gee
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.cov_struct import Exchangeable
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/mnt/user-data/uploads'
OUT_DIR  = '/mnt/user-data/outputs'
RADIUS   = 1.0  # degrees (~111 km)


In [127]:
print("=" * 70)
print("DATA LOADING AND SITE-SPECIFIC MP MATCHING")
print("=" * 70)

oy_url = 'https://raw.githubusercontent.com/dhairya-p/bc3412-code/main/data/Oyster%20DTA/Oyster_Histopath%20(1995-2010).csv'
mp_url = 'https://raw.githubusercontent.com/dhairya-p/bc3412-code/main/data/Microplastics%20Concentration/Marine_Microplastics_Oysters%20(1989-2016).csv'

oy = pd.read_csv(oy_url)
# oy['Sample Date'] = pd.to_datetime(oy['Sample Date'], format='mixed', errors='coerce') # Convert to datetime - REMOVED DUE TO KeyError
# oy['month'] = oy['Sample Date'].dt.month # Extract month - REMOVED DUE TO KeyError

mp = pd.read_csv(mp_url)
mp['Sample Date'] = pd.to_datetime(mp['Sample Date'], format='mixed')
mp['year']  = mp['Sample Date'].dt.year
mp['month'] = mp['Sample Date'].dt.month

print(f"Oysters:  {len(oy):,} records, {oy['Fiscal Year'].min()}–{oy['Fiscal Year'].max()}")
print(f"  DTA recorded: {oy['Digestive Tubule Atrophy'].notna().sum()}")
print(f"MP (new): {len(mp):,} records, all ocean surface water")

DATA LOADING AND SITE-SPECIFIC MP MATCHING
Oysters:  3,638 records, 1988–2010
  DTA recorded: 1100
MP (new): 3,191 records, all ocean surface water


In [128]:
# =============================================================
# MERGE WATER QUALITY DATA
# =============================================================
print(f"\n{'=' * 70}")
print("MERGING WATER QUALITY DATA")
print("=" * 70)

# Load water quality data (moved from cell 7e99732f to resolve dependency)
water_quality_url = 'https://raw.githubusercontent.com/dhairya-p/bc3412-code/main/water_quality.csv'
water_quality_df = pd.read_csv(water_quality_url)
print(f"Water Quality: {len(water_quality_df):,} records")

# Aggregate water_quality_df by year to match oy_dta_regional's annual granularity
# This assumes annual average water quality is relevant
annual_water_quality = water_quality_df.groupby('year').mean().reset_index()

# Merge with oy_dta_regional on Fiscal Year
oy_dta_regional_wq = oy_dta_regional.merge(
    annual_water_quality.rename(columns={'year': 'Fiscal Year'}),
    on='Fiscal Year',
    how='left' # Use left join to keep all oyster data
)

print(f"\nOysters with DTA + regional MP + Water Quality: {len(oy_dta_regional_wq):,} records")
print(f"Unique years in merged data: {oy_dta_regional_wq['Fiscal Year'].nunique()}")
print(f"First 5 rows of merged data:\n{oy_dta_regional_wq.head()}")
print(f"Columns in merged data: {oy_dta_regional_wq.columns.tolist()}")


MERGING WATER QUALITY DATA
Water Quality: 144 records

Oysters with DTA + regional MP + Water Quality: 985 records
Unique years in merged data: 14
First 5 rows of merged data:
          Study NST_Site General Location Specific Location State Name  \
0  Mussel Watch     BBGC     Biscayne Bay     Gould's Canal    Florida   
1  Mussel Watch     BBGC     Biscayne Bay     Gould's Canal    Florida   
2  Mussel Watch     BBGC     Biscayne Bay     Gould's Canal    Florida   
3  Mussel Watch     BBGC     Biscayne Bay     Gould's Canal    Florida   
4  Mussel Watch     BBGC     Biscayne Bay     Gould's Canal    Florida   

       Region                Specific Region         Coastal Ecological Area  \
0  East Coast  Florida Keys to Cape Hatteras  Central and South Biscayne Bay   
1  East Coast  Florida Keys to Cape Hatteras  Central and South Biscayne Bay   
2  East Coast  Florida Keys to Cape Hatteras  Central and South Biscayne Bay   
3  East Coast  Florida Keys to Cape Hatteras  Central and 

In [129]:
print(oy.columns)

Index(['Study', 'NST_Site', 'General Location', 'Specific Location',
       'State Name', 'Region', 'Specific Region', 'Coastal Ecological Area',
       'Latitude', 'Longitude', 'NST_Sample_ID', 'Station Letter',
       'Sample Number', 'Fiscal Year', 'Condition Code',
       'Condition Code Description', 'Sex', 'Gonadal Index',
       'Gonadal Index Description', 'Length', 'Wet Weight',
       'Full Displacement Volume', 'Empty Displacement Volume', 'Dermo',
       'Dermo Infection Intensity', 'Dermo Numerical Value',
       'Dermo Description', 'Gonad Subsample Wet Weight', 'Ceroid',
       'Cestode Body', 'Cestode Gill', 'Cestode Mantle',
       'Ciliate Digestive Tract', 'Ciliate Gut', 'Ciliate Large Gill',
       'Ciliate Small Gill', 'Copepod Body', 'Copepod Gill',
       'Copepod Gut Digestive Tubule', 'Nematode', 'Nematopsis Body',
       'Nematopsis Gill', 'Nematopsis Mantle', 'Neoplasm', 'Pea Crab',
       'Rickettsia Digestive Tubule', 'Rickettsia Gut', 'Proctoeces',
       

In [130]:
# =============================================================
# SITE-SPECIFIC MP MATCHING (1° radius)
# =============================================================
print(f"\nMatching MP readings within {RADIUS}° (~{RADIUS*111:.0f} km) of each oyster site...")

# Get unique oyster sites
sites = oy.groupby(['General Location','Latitude','Longitude','State Name']).agg(
    n_oysters=('NST_Sample_ID','count'),
    year_min=('Fiscal Year','min'),
    year_max=('Fiscal Year','max'),
    n_dta=('Digestive Tubule Atrophy', lambda x: x.notna().sum())
).reset_index()

# For each oyster site, find nearby MP and compute annual means
site_mp_list = []
for _, site in sites.iterrows():
    lat, lon = site['Latitude'], site['Longitude']
    nearby = mp[
        (abs(mp['Latitude (degree)'] - lat) <= RADIUS) &
        (abs(mp['Longitude (degree)'] - lon) <= RADIUS)
    ].copy()
    if len(nearby) > 0:
        nearby['oyster_site'] = site['General Location']
        nearby['oyster_lat']  = lat
        nearby['oyster_lon']  = lon
        nearby['oyster_state'] = site['State Name']
        site_mp_list.append(nearby)

site_mp = pd.concat(site_mp_list, ignore_index=True)

# Compute annual mean MP per oyster site
site_mp_annual = site_mp.groupby(['oyster_site','oyster_lat','oyster_lon','oyster_state','year']).agg(
    mp_mean  = ('Microplastics Measurement','mean'),
    mp_count = ('Microplastics Measurement','count'),
    mp_std   = ('Microplastics Measurement','std')
).reset_index()

# Also compute a regional annual mean (across all sites for that year)
regional_mp = site_mp.groupby('year').agg(
    mp_regional_mean  = ('Microplastics Measurement','mean'),
    mp_regional_count = ('Microplastics Measurement','count')
).reset_index()

print(f"\nSite-specific MP annual records: {len(site_mp_annual)}")
print(f"Regional annual MP (all sites pooled): {len(regional_mp)} years")



Matching MP readings within 1.0° (~111 km) of each oyster site...

Site-specific MP annual records: 124
Regional annual MP (all sites pooled): 22 years


In [131]:
# =============================================================
# JOIN MP TO INDIVIDUAL OYSTERS
# =============================================================
print(f"\n{'=' * 70}")
print("JOINING MP DATA TO INDIVIDUAL OYSTERS")
print("=" * 70)

# Strategy A: Site-specific join (each oyster gets MP from its own site's neighbourhood)
oy_model = oy.copy()
oy_model = oy_model.merge(
    site_mp_annual[['oyster_site','year','mp_mean','mp_count']].rename(
        columns={'oyster_site':'General Location','year':'Fiscal Year'}
    ),
    on=['General Location','Fiscal Year'],
    how='inner'
)

# Strategy B: Regional join (each oyster gets the regional annual mean)
oy_regional = oy.copy()
oy_regional = oy_regional.merge(
    regional_mp.rename(columns={'year':'Fiscal Year'}),
    on='Fiscal Year',
    how='inner'
)

# Filter to those with DTA
oy_dta = oy_model[oy_model['Digestive Tubule Atrophy'].notna()].copy()
oy_dta['dta'] = oy_dta['Digestive Tubule Atrophy'].astype(int)
oy_dta['severe'] = (oy_dta['dta'] >= 3).astype(int)

oy_dta_regional = oy_regional[oy_regional['Digestive Tubule Atrophy'].notna()].copy()
oy_dta_regional['dta'] = oy_dta_regional['Digestive Tubule Atrophy'].astype(int)
oy_dta_regional['severe'] = (oy_dta_regional['dta'] >= 3).astype(int)

print(f"\nStrategy A (site-specific MP):")
print(f"  Oysters with DTA + matched MP: {len(oy_dta)}")
print(f"  Unique years: {oy_dta['Fiscal Year'].nunique()}")
print(f"  Unique sites: {oy_dta['General Location'].nunique()}")
print(f"  Unique MP values: {oy_dta['mp_mean'].nunique()}")

print(f"\nStrategy B (regional MP):")
print(f"  Oysters with DTA + regional MP: {len(oy_dta_regional)}")
print(f"  Unique years: {oy_dta_regional['Fiscal Year'].nunique()}")
print(f"  Unique MP values: {oy_dta_regional['mp_regional_mean'].nunique()}")

print(f"\nDTA distribution in matched data:")
for v in sorted(oy_dta['dta'].unique()):
    n = (oy_dta['dta']==v).sum()
    desc = {0:'Normal',1:'Slight',2:'Half normal',3:'Significant',4:'Extreme'}
    print(f"  {v} ({desc.get(v,'?'):12s}): {n:>4} ({100*n/len(oy_dta):.1f}%)")



JOINING MP DATA TO INDIVIDUAL OYSTERS

Strategy A (site-specific MP):
  Oysters with DTA + matched MP: 125
  Unique years: 8
  Unique sites: 7
  Unique MP values: 19

Strategy B (regional MP):
  Oysters with DTA + regional MP: 985
  Unique years: 14
  Unique MP values: 14

DTA distribution in matched data:
  0 (Normal      ):    3 (2.4%)
  1 (Slight      ):   31 (24.8%)
  2 (Half normal ):   37 (29.6%)
  3 (Significant ):   41 (32.8%)
  4 (Extreme     ):   13 (10.4%)


In [132]:
# Summary by year
print(f"\nAnnual summary (site-specific match):")
yr_sum = oy_dta.groupby('Fiscal Year').agg(
    n_oysters=('dta','count'),
    dta_median=('dta','median'),
    dta_mean=('dta','mean'),
    pct_severe=('severe','mean'),
    mp_mean=('mp_mean','mean'),
).reset_index()
yr_sum['pct_severe'] = yr_sum['pct_severe'] * 100
print(f"{'Year':>6} {'n':>5} {'DTA med':>8} {'DTA mean':>9} {'% Severe':>10} {'MP mean':>12}")
print("-" * 50)
for _, r in yr_sum.iterrows():
    print(f"{int(r['Fiscal Year']):>6} {int(r['n_oysters']):>5} {r['dta_median']:>8.1f}"
          f" {r['dta_mean']:>9.2f} {r['pct_severe']:>10.1f} {r['mp_mean']:>12.6f}")



Annual summary (site-specific match):
  Year     n  DTA med  DTA mean   % Severe      MP mean
--------------------------------------------------
  1995    15      3.0      2.73       86.7     0.006510
  1997    15      2.0      2.13       33.3     0.009620
  1999    15      2.0      2.33       46.7     0.023711
  2001    15      3.0      2.33       53.3     0.004720
  2003    10      2.0      2.00       30.0     0.002250
  2005    25      2.0      1.84       16.0     0.006555
  2007    10      2.0      2.10       30.0     0.002067
  2008    20      3.0      2.50       55.0     0.001350


In [133]:
# =============================================================
# STEP 1: SPEARMAN ON AGGREGATED ANNUAL POINTS (Honest n)
# =============================================================
print(f"\n{'=' * 70}")
print("STEP 1: Spearman Correlation on Aggregated Annual Points")
print("=" * 70)

# Use regional MP for cleaner annual aggregate
yr_agg = oy_dta_regional.groupby('Fiscal Year').agg(
    dta_median=('dta','median'),
    dta_mean=('dta','mean'),
    pct_severe=('severe','mean'),
    n_oysters=('dta','count'),
    mp_mean=('mp_regional_mean','first')
).reset_index()
yr_agg['pct_severe'] = yr_agg['pct_severe'] * 100

n_years = len(yr_agg)
rho_mean, p_mean = stats.spearmanr(yr_agg['mp_mean'], yr_agg['dta_mean'])
rho_sev,  p_sev  = stats.spearmanr(yr_agg['mp_mean'], yr_agg['pct_severe'])
rho_med,  p_med  = stats.spearmanr(yr_agg['mp_mean'], yr_agg['dta_median'])

print(f"\nn = {n_years} years")
print(f"\n  MP vs Mean DTA:     ρ = {rho_mean:+.3f}, p = {p_mean:.4f}")
print(f"  MP vs % Severe:     ρ = {rho_sev:+.3f}, p = {p_sev:.4f}")
print(f"  MP vs Median DTA:   ρ = {rho_med:+.3f}, p = {p_med:.4f}")


# =============================================================
# STEP 2: ORDINAL LOGISTIC — MP Only
# =============================================================
print(f"\n{'=' * 70}")
print("STEP 2: Ordinal Logistic — DTA ~ MP Concentration")
print("=" * 70)

# Use regional MP (cleaner, one value per year)
X_mp = oy_dta_regional[['mp_regional_mean']].reset_index(drop=True)
y_dta = oy_dta_regional['dta'].astype(int).reset_index(drop=True)

ord_mp = OrderedModel(y_dta, X_mp, distr='logit').fit(method='bfgs', disp=0)
mp_coef = ord_mp.params['mp_regional_mean']
mp_or   = np.exp(mp_coef)
mp_p    = ord_mp.pvalues['mp_regional_mean']

print(f"\n  n = {len(oy_dta_regional):,} oysters ({oy_dta_regional['mp_regional_mean'].nunique()} unique MP values)")
print(f"  MP coef = {mp_coef:.4f}, OR = {mp_or:.4f}, p = {mp_p:.6f}")
print(f"  Direction: {'Higher MP → more atrophy' if mp_coef > 0 else 'Higher MP → less atrophy'}")
print(f"\n  ⚠ P-value inflated: {len(oy_dta_regional):,} rows but only"
      f" {oy_dta_regional['mp_regional_mean'].nunique()} unique MP values")


# =============================================================
# STEP 3: ORDINAL LOGISTIC — MP + Location
# =============================================================
print(f"\n{'=' * 70}")
print("STEP 3: Ordinal Logistic — DTA ~ MP + Location")
print("=" * 70)

loc_dum = pd.get_dummies(oy_dta_regional['General Location'], prefix='loc', drop_first=True).astype(float)
X_mp_loc = pd.concat([
    oy_dta_regional[['mp_regional_mean']].reset_index(drop=True),
    loc_dum.reset_index(drop=True)
], axis=1)

ord_mp_loc = OrderedModel(y_dta, X_mp_loc, distr='logit').fit(method='bfgs', disp=0)
mp_coef2 = ord_mp_loc.params['mp_regional_mean']
mp_or2   = np.exp(mp_coef2)
mp_p2    = ord_mp_loc.pvalues['mp_regional_mean']

print(f"\n  MP coef = {mp_coef2:.4f}, OR = {mp_or2:.4f}, p = {mp_p2:.6f}")
print(f"  Direction: {'Higher MP → more atrophy' if mp_coef2 > 0 else 'Higher MP → less atrophy'}")
print(f"  AIC without location: {ord_mp.aic:.1f}")
print(f"  AIC with location:    {ord_mp_loc.aic:.1f}")


# =============================================================
# STEP 4: GEE — Cluster-Robust Standard Errors (Honest Test)
# =============================================================
print(f"\n{'=' * 70}")
print("STEP 4: GEE — Cluster-Robust vs Naive (Binary: Severe DTA)")
print("=" * 70)

gee_df = oy_dta_regional[['severe','mp_regional_mean','Fiscal Year','General Location']].dropna().copy()
gee_df = gee_df.sort_values('Fiscal Year')

# Naive logistic
naive = smf.logit('severe ~ mp_regional_mean', data=gee_df).fit(disp=0)

# GEE clustered by year
gee_year = gee.GEE.from_formula(
    'severe ~ mp_regional_mean',
    groups='Fiscal Year', data=gee_df,
    family=Binomial(), cov_struct=Exchangeable()
).fit()

se_naive = naive.bse['mp_regional_mean']
se_gee   = gee_year.bse['mp_regional_mean']
se_inflation = se_gee / se_naive

print(f"\n{'Method':<40} {'Coef':>10} {'SE':>10} {'p-value':>10} {'OR':>10}")
print("-" * 80)
print(f"{'Naive logistic (inflated n):':<40} {naive.params['mp_regional_mean']:>10.4f}"
      f" {se_naive:>10.4f} {naive.pvalues['mp_regional_mean']:>10.6f}"
      f" {np.exp(naive.params['mp_regional_mean']):>10.4f}")
print(f"{'GEE clustered by year (honest):':<40} {gee_year.params['mp_regional_mean']:>10.4f}"
      f" {se_gee:>10.4f} {gee_year.pvalues['mp_regional_mean']:>10.6f}"
      f" {np.exp(gee_year.params['mp_regional_mean']):>10.4f}")

naive_sig = naive.pvalues['mp_regional_mean'] < 0.05
gee_sig   = gee_year.pvalues['mp_regional_mean'] < 0.05
n_clusters = gee_df['Fiscal Year'].nunique()

print(f"\n  Clusters (effective n): {n_clusters}")
print(f"  SE inflation: {se_inflation:.1f}×")
print(f"  Naive significant:  {'Yes' if naive_sig else 'No'}")
print(f"  GEE significant:    {'Yes' if gee_sig else 'No'}")


# =============================================================
# STEP 5: ROBUSTNESS — Condition Code & Dermo
# =============================================================
print(f"\n{'=' * 70}")
print("STEP 5: Robustness Checks — Condition Code & Dermo")
print("=" * 70)

# Condition Code
oy_cc = oy_regional[oy_regional['Condition Code'].notna()].copy()
oy_cc['cc'] = oy_cc['Condition Code'].astype(int)
oy_cc_clean = oy_cc[['cc','mp_regional_mean','Fiscal Year']].dropna()

if len(oy_cc_clean) > 10:
    X_cc = oy_cc_clean[['mp_regional_mean']].reset_index(drop=True)
    y_cc = oy_cc_clean['cc'].astype(int).reset_index(drop=True)
    ord_cc = OrderedModel(y_cc, X_cc, distr='logit').fit(method='bfgs', disp=0)
    print(f"\nCondition Code ~ MP:")
    print(f"  n = {len(oy_cc_clean):,}, unique years = {oy_cc_clean['Fiscal Year'].nunique()}")
    print(f"  MP coef = {ord_cc.params['mp_regional_mean']:.4f},"
          f" OR = {np.exp(ord_cc.params['mp_regional_mean']):.4f},"
          f" p = {ord_cc.pvalues['mp_regional_mean']:.6f}")
    cc_coef = ord_cc.params['mp_regional_mean']
    cc_p = ord_cc.pvalues['mp_regional_mean']
else:
    print("\nCondition Code: insufficient matched data")
    cc_coef = cc_p = np.nan

# Dermo
oy_dm = oy_regional[oy_regional['Dermo Numerical Value'].notna()].copy()
oy_dm_clean = oy_dm[['Dermo Numerical Value','mp_regional_mean','Fiscal Year']].dropna()

if len(oy_dm_clean) > 10:
    dermo_ols = smf.ols('Q("Dermo Numerical Value") ~ mp_regional_mean', data=oy_dm_clean).fit()
    print(f"\nDermo Numerical Value ~ MP (OLS):")
    print(f"  n = {len(oy_dm_clean):,}, unique years = {oy_dm_clean['Fiscal Year'].nunique()}")
    print(f"  MP coef = {dermo_ols.params['mp_regional_mean']:.4f},"
          f" p = {dermo_ols.pvalues['mp_regional_mean']:.6f}")
    print(f"  R² = {dermo_ols.rsquared:.4f}")
    dermo_coef = dermo_ols.params['mp_regional_mean']
    dermo_p = dermo_ols.pvalues['mp_regional_mean']

    # GEE for Dermo too
    dm_gee_df = oy_dm_clean.copy().sort_values('Fiscal Year')
    dm_gee_df['dermo_binary'] = (dm_gee_df['Dermo Numerical Value'] > 0).astype(int)
    dm_naive = smf.logit('dermo_binary ~ mp_regional_mean', data=dm_gee_df).fit(disp=0)
    dm_gee = gee.GEE.from_formula(
        'dermo_binary ~ mp_regional_mean',
        groups='Fiscal Year', data=dm_gee_df,
        family=Binomial(), cov_struct=Exchangeable()
    ).fit()
    print(f"\n  Dermo (binary: infected vs not) — GEE:")
    print(f"    Naive p = {dm_naive.pvalues['mp_regional_mean']:.6f}")
    print(f"    GEE   p = {dm_gee.pvalues['mp_regional_mean']:.6f}")
    dermo_gee_p = dm_gee.pvalues['mp_regional_mean']
else:
    print("\nDermo: insufficient matched data")
    dermo_coef = dermo_p = dermo_gee_p = np.nan


# =============================================================
# STEP 6: PROPORTIONAL ODDS CHECK (DTA model)
# =============================================================
print(f"\n{'=' * 70}")
print("STEP 6: Proportional Odds Assumption Check")
print("=" * 70)

X_po = oy_dta_regional[['mp_regional_mean']].reset_index(drop=True)
print(f"\n{'Threshold':<25} {'MP Coef':>10} {'p-value':>10}")
print("-" * 45)
po_coefs = []
for cut in [0, 1, 2, 3]:
    y_bin = (oy_dta_regional['dta'].reset_index(drop=True) <= cut).astype(int)
    if y_bin.sum() > 5 and (1-y_bin).sum() > 5:
        lg = Logit(y_bin, X_po).fit(disp=0)
        po_coefs.append(lg.params['mp_regional_mean'])
        print(f"DTA ≤ {cut} vs > {cut}          {lg.params['mp_regional_mean']:>10.4f}"
              f" {lg.pvalues['mp_regional_mean']:>10.6f}")

if len(po_coefs) >= 2:
    ratios = [po_coefs[i]/po_coefs[0] if po_coefs[0] != 0 else float('inf')
              for i in range(1, len(po_coefs))]
    print(f"\n  Coefficient ratios (vs first threshold): {[f'{r:.2f}' for r in ratios]}")
    violated = any(abs(r - 1) > 0.5 for r in ratios)
    print(f"  Proportional odds: {'⚠ VIOLATED' if violated else '✓ OK'}")


# =============================================================
# STEP 7: FLORIDA-ONLY ROBUSTNESS
# =============================================================
print(f"\n{'=' * 70}")
print("STEP 7: Florida-Only Subanalysis (Best Spatial Overlap)")
print("=" * 70)

fl_dta = oy_dta[oy_dta['State Name'] == 'Florida'].copy()
if len(fl_dta) >= 20 and fl_dta['mp_mean'].nunique() >= 3:
    X_fl = fl_dta[['mp_mean']].reset_index(drop=True)
    y_fl = fl_dta['dta'].astype(int).reset_index(drop=True)
    ord_fl = OrderedModel(y_fl, X_fl, distr='logit').fit(method='bfgs', disp=0)

    print(f"\n  Florida oysters with site-matched MP: {len(fl_dta)}")
    print(f"  Unique years: {fl_dta['Fiscal Year'].nunique()}")
    print(f"  Unique MP values: {fl_dta['mp_mean'].nunique()}")
    print(f"  MP coef = {ord_fl.params['mp_mean']:.4f},"
          f" OR = {np.exp(ord_fl.params['mp_mean']):.4f},"
          f" p = {ord_fl.pvalues['mp_mean']:.6f}")
    fl_coef = ord_fl.params['mp_mean']
    fl_p = ord_fl.pvalues['mp_mean']

    # GEE for Florida
    fl_gee_df = fl_dta[['severe','mp_mean','Fiscal Year']].dropna().sort_values('Fiscal Year')
    if fl_gee_df['Fiscal Year'].nunique() >= 3:
        fl_naive = smf.logit('severe ~ mp_mean', data=fl_gee_df).fit(disp=0)
        fl_gee = gee.GEE.from_formula(
            'severe ~ mp_mean', groups='Fiscal Year', data=fl_gee_df,
            family=Binomial(), cov_struct=Exchangeable()
        ).fit()
        fl_se_inf = fl_gee.bse['mp_mean'] / fl_naive.bse['mp_mean']
        print(f"\n  GEE (Florida, clustered by year):")
        print(f"    Naive p = {fl_naive.pvalues['mp_mean']:.6f}")
        print(f"    GEE   p = {fl_gee.pvalues['mp_mean']:.6f}")
        print(f"    SE inflation: {fl_se_inf:.1f}×")
        fl_gee_p = fl_gee.pvalues['mp_mean']
    else:
        fl_gee_p = np.nan
else:
    print("  Insufficient Florida data for subanalysis")
    fl_coef = fl_p = fl_gee_p = np.nan


# =============================================================
# VISUALISATIONS
# =============================================================
print(f"\n{'=' * 70}")
print("GENERATING VISUALISATIONS...")
print("=" * 70)

fig = plt.figure(figsize=(18, 14))
gs  = fig.add_gridspec(3, 3, hspace=0.5, wspace=0.35)

# --- R1C1: Annual scatter — MP vs DTA ---
ax = fig.add_subplot(gs[0, 0])
ax.scatter(yr_agg['mp_mean'], yr_agg['dta_mean'], s=100, color='steelblue',
           edgecolors='black', lw=0.8, zorder=5)
for _, r in yr_agg.iterrows():
    ax.annotate(str(int(r['Fiscal Year'])), (r['mp_mean'], r['dta_mean']),
                textcoords='offset points', xytext=(7, 4), fontsize=7)
ax.set_xlabel('Regional MP Concentration')
ax.set_ylabel('Mean DTA Score')
ax.set_title(f'Step 1: Annual Correlation (n={n_years})\n'
             f'Spearman ρ={rho_mean:+.3f}, p={p_mean:.4f}', fontweight='bold')
ax.grid(True, alpha=0.3)

# --- R1C2: Annual scatter — MP vs % Severe ---
ax = fig.add_subplot(gs[0, 1])
ax.scatter(yr_agg['mp_mean'], yr_agg['pct_severe'], s=100, color='coral',
           edgecolors='black', lw=0.8, zorder=5)
for _, r in yr_agg.iterrows():
    ax.annotate(str(int(r['Fiscal Year'])), (r['mp_mean'], r['pct_severe']),
                textcoords='offset points', xytext=(7, 4), fontsize=7)
ax.set_xlabel('Regional MP Concentration')
ax.set_ylabel('% Severe DTA (score ≥ 3)')
ax.set_title(f'Step 1: MP vs % Severe DTA\n'
             f'Spearman ρ={rho_sev:+.3f}, p={p_sev:.4f}', fontweight='bold')
ax.grid(True, alpha=0.3)

# --- R1C3: Dual time series ---
ax = fig.add_subplot(gs[0, 2])
ax2 = ax.twinx()
ax.plot(yr_agg['Fiscal Year'], yr_agg['dta_mean'], 'o-', color='steelblue',
        lw=2, ms=6, label='Mean DTA')
ax2.plot(yr_agg['Fiscal Year'], yr_agg['mp_mean'], 's--', color='coral',
         lw=2, ms=6, label='MP Conc')
ax.set_xlabel('Year')
ax.set_ylabel('Mean DTA', color='steelblue')
ax2.set_ylabel('MP Concentration', color='coral')
ax.set_title('Time Series: DTA and MP\nOver Overlapping Years', fontweight='bold')
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, fontsize=8, loc='upper left')
ax.grid(True, alpha=0.2)

# --- R2C1: Naive vs GEE ---
ax = fig.add_subplot(gs[1, 0])
pv = [naive.pvalues['mp_regional_mean'], gee_year.pvalues['mp_regional_mean']]
cols = ['#f44336' if p < 0.05 else '#4CAF50' for p in pv]
bars = ax.bar(['Naive Logistic\n(inflated n)', f'GEE Clustered\n({n_clusters} years)'],
              pv, color=cols, edgecolor='black', lw=0.8, width=0.5)
ax.axhline(0.05, color='red', ls='--', lw=1.5, label='α = 0.05')
for b, p_val in zip(bars, pv):
    ax.text(b.get_x()+b.get_width()/2, p_val+0.02,
            f'p={p_val:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('p-value')
ax.set_ylim(0, max(pv)*1.5)
ax.set_title(f'Step 4: Naive vs Cluster-Robust\nSE inflates {se_inflation:.1f}×', fontweight='bold')
ax.legend(fontsize=9)

# --- R2C2: Model comparison ---
ax = fig.add_subplot(gs[1, 1])
model_labels = ['MP only\n(ordinal)', 'MP + Location\n(ordinal)', 'Severe DTA\n(GEE)']
model_coefs = [mp_coef, mp_coef2, gee_year.params['mp_regional_mean']]
model_pvals = [mp_p, mp_p2, gee_year.pvalues['mp_regional_mean']]
m_cols = ['#f44336' if p < 0.05 else '#9E9E9E' for p in model_pvals]
bars = ax.bar(model_labels, model_coefs, color=m_cols, edgecolor='black', lw=0.8, width=0.55)
ax.axhline(0, color='black', lw=0.8)
for b, c, pv in zip(bars, model_coefs, model_pvals):
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else 'ns'
    yp = c + 0.3 if c > 0 else c - 0.3
    ax.text(b.get_x()+b.get_width()/2, yp, f'{c:.2f}\n({sig})',
            ha='center', va='bottom' if c > 0 else 'top', fontsize=8)
ax.set_ylabel('MP Coefficient')
ax.set_title('Step 2–4: MP Effect Across Models', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# --- R2C3: Robustness across DVs ---
ax = fig.add_subplot(gs[1, 2])
rob_labels = ['DTA\n(ordinal)', 'Condition\nCode', 'Dermo\n(OLS)']
rob_pvals = [mp_p, cc_p if not np.isnan(cc_p) else 1.0,
             dermo_p if not np.isnan(dermo_p) else 1.0]
rob_cols = ['#f44336' if p < 0.05 else '#4CAF50' for p in rob_pvals]
bars = ax.bar(rob_labels, rob_pvals, color=rob_cols, edgecolor='black', lw=0.8, width=0.5)
ax.axhline(0.05, color='red', ls='--', lw=1.5, label='α = 0.05')
for b, pv in zip(bars, rob_pvals):
    ax.text(b.get_x()+b.get_width()/2, pv+0.02,
            f'p={pv:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('p-value (naive, ordinal/OLS)')
ax.set_title('Step 5: Robustness — Different DVs', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, max(rob_pvals)*1.5)

# --- R3C1: DTA distribution by MP tertile ---
ax = fig.add_subplot(gs[2, 0])
oy_dta_regional['mp_tertile'] = pd.qcut(
    oy_dta_regional['mp_regional_mean'], q=3, labels=['Low MP','Mid MP','High MP']
)
for i, tert in enumerate(['Low MP','Mid MP','High MP']):
    sub = oy_dta_regional[oy_dta_regional['mp_tertile']==tert]
    counts = [100*(sub['dta']==v).mean() for v in range(5)]
    x = np.arange(5) + i*0.25
    colors = ['#4CAF50','#8BC34A','#FFC107','#FF9800','#f44336']
    ax.bar(x, counts, width=0.22, label=tert, color=colors[i], edgecolor='black', lw=0.3)
ax.set_xticks(np.arange(5)+0.25)
ax.set_xticklabels(['0\nNormal','1\nSlight','2\nHalf','3\nSignif.','4\nExtreme'])
ax.set_ylabel('% of Oysters')
ax.set_title('DTA Distribution by MP Tertile', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# --- R3C2: MP concentration over time in oyster region ---
ax = fig.add_subplot(gs[2, 1])
ax.scatter(regional_mp['year'], regional_mp['mp_regional_mean'], s=60, color='coral',
           edgecolors='black', lw=0.5, zorder=5)
ax.plot(regional_mp['year'], regional_mp['mp_regional_mean'], '-', color='coral', alpha=0.4)
ax.set_xlabel('Year')
ax.set_ylabel('Regional MP Concentration')
ax.set_title('MP Concentration in Oyster Region\nOver Time', fontweight='bold')
ax.grid(True, alpha=0.3)

# --- R3C3: Spatial coverage ---
ax = fig.add_subplot(gs[2, 2])
# Plot oyster sites
for _, site in sites.iterrows():
    ax.scatter(site['Longitude'], site['Latitude'], s=60, c='blue',
               edgecolors='black', lw=0.3, zorder=5)
# Plot MP readings
mp_oy_region = mp[
    (mp['Latitude (degree)'] >= 24) & (mp['Latitude (degree)'] <= 41) &
    (mp['Longitude (degree)'] >= -83) & (mp['Longitude (degree)'] <= -74)
]
ax.scatter(mp_oy_region['Longitude (degree)'], mp_oy_region['Latitude (degree)'],
           s=8, c='red', alpha=0.3, zorder=3)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Spatial Coverage\nBlue=Oyster, Red=MP', fontweight='bold')
ax.grid(True, alpha=0.2)

fig.suptitle('Does Microplastic Concentration Affect Oyster Digestive Tubule Atrophy?\n'
             'Site-Specific Analysis — BC3412 Consulting Project',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig(f'{OUT_DIR}/oyster_dta_mp_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
print(f"  Saved: oyster_dta_mp_analysis.png")
plt.close()


STEP 1: Spearman Correlation on Aggregated Annual Points

n = 14 years

  MP vs Mean DTA:     ρ = +0.437, p = 0.1178
  MP vs % Severe:     ρ = +0.204, p = 0.4833
  MP vs Median DTA:   ρ = +0.278, p = 0.3363

STEP 2: Ordinal Logistic — DTA ~ MP Concentration

  n = 985 oysters (14 unique MP values)
  MP coef = 15.4962, OR = 5369150.9609, p = 0.119404
  Direction: Higher MP → more atrophy

  ⚠ P-value inflated: 985 rows but only 14 unique MP values

STEP 3: Ordinal Logistic — DTA ~ MP + Location

  MP coef = 5.7781, OR = 323.1293, p = 0.572384
  Direction: Higher MP → more atrophy
  AIC without location: 2574.1
  AIC with location:    2445.6

STEP 4: GEE — Cluster-Robust vs Naive (Binary: Severe DTA)

Method                                         Coef         SE    p-value         OR
--------------------------------------------------------------------------------
Naive logistic (inflated n):                15.2085    11.0175   0.167466 4026964.6080
GEE clustered by year (honest):      

### STEP 8: Ordinal Logistic — DTA ~ MP + Water Quality

Let's build an ordinal logistic model that includes selected water quality parameters in addition to microplastic concentration.

In [134]:
print(f"\n{'=' * 70}")
print("STEP 8: Ordinal Logistic — DTA ~ MP + Water Quality")
print("=" * 70)

# Drop rows with NaN values in the selected columns for the model
# We'll use oy_dta_regional_wq which contains the merged water quality data
model_df_ord_wq = oy_dta_regional_wq[['dta', 'mp_regional_mean', 'ph', 'no3', 'so', 'Fiscal Year']].dropna().copy()

X_mp_wq = model_df_ord_wq[['mp_regional_mean', 'ph', 'no3', 'so']].reset_index(drop=True)
y_dta_wq = model_df_ord_wq['dta'].astype(int).reset_index(drop=True)

if not X_mp_wq.empty and not y_dta_wq.empty:
    print(f"X_mp_wq unique values per column before model fit:\n{X_mp_wq.nunique()}")
    print(f"X_mp_wq standard deviation per column before model fit:\n{X_mp_wq.std()}")
    print(f"Unique Fiscal Years in model_df_ord_wq: {model_df_ord_wq['Fiscal Year'].nunique()}")
    print(f"Number of unique covariate patterns in X_mp_wq: {len(X_mp_wq.drop_duplicates())}")

    # Check for and remove constant columns from X_mp_wq
    # Using nunique() <= 1 is often more robust for constant checks than std() == 0 due to float precision.
    constant_cols = [col for col in X_mp_wq.columns if X_mp_wq[col].nunique() <= 1]
    if constant_cols:
        print(f"Warning: The following columns are constant after dropping NaNs and will be removed from the OrderedModel: {constant_cols}")
        X_mp_wq = X_mp_wq.drop(columns=constant_cols)

    # Ensure X_mp_wq is not empty after removing constant columns
    if X_mp_wq.empty:
        print("  All predictor columns became constant or were removed. Cannot fit Ordinal Logistic model.")
    else:
        ord_mp_wq = OrderedModel(y_dta_wq, X_mp_wq, distr='logit').fit(method='bfgs', disp=0)

        # Make sure to handle cases where 'mp_regional_mean' itself might have been dropped as a constant
        mp_coef_wq = ord_mp_wq.params['mp_regional_mean'] if 'mp_regional_mean' in ord_mp_wq.params else np.nan
        mp_or_wq   = np.exp(mp_coef_wq) if not np.isnan(mp_coef_wq) else np.nan
        mp_p_wq    = ord_mp_wq.pvalues['mp_regional_mean'] if 'mp_regional_mean' in ord_mp_wq.pvalues else np.nan

        print(f"\n  n = {len(model_df_ord_wq):,} oysters (considering water quality data)")
        print(f"  MP coef = {mp_coef_wq:.4f}, OR = {mp_or_wq:.4f}, p = {mp_p_wq:.6f}")
        print(f"  Direction (MP): {'Higher MP → more atrophy' if (not np.isnan(mp_coef_wq) and mp_coef_wq > 0) else 'Higher MP → less atrophy'}")
        print(f"\n  Water Quality Parameters:")
        for param in ['ph', 'no3', 'so']:
            if param in X_mp_wq.columns and param in ord_mp_wq.params: # Check if param exists in X_mp_wq first
                coef = ord_mp_wq.params[param]
                or_val = np.exp(coef)
                p_val = ord_mp_wq.pvalues[param]
                print(f"    {param} coef = {coef:.4f}, OR = {or_val:.4f}, p = {p_val:.6f}")
                print(f"    Direction ({param}): {'Higher value → more atrophy' if coef > 0 else 'Higher value → less atrophy'}")
        print(f"  AIC with MP only (from Step 2): {ord_mp.aic:.1f}")
        print(f"  AIC with MP + Water Quality:    {ord_mp_wq.aic:.1f}")
else:
    print("  Insufficient data to run Ordinal Logistic with Water Quality.")

# Store for plotting later
mp_coef_wq_val = mp_coef_wq if 'mp_coef_wq' in locals() else np.nan
mp_p_wq_val = mp_p_wq if 'mp_p_wq' in locals() else np.nan


STEP 8: Ordinal Logistic — DTA ~ MP + Water Quality
X_mp_wq unique values per column before model fit:
mp_regional_mean    5
ph                  5
no3                 5
so                  5
dtype: int64
X_mp_wq standard deviation per column before model fit:
mp_regional_mean    0.003539
ph                  0.004589
no3                 0.088659
so                  0.035871
dtype: float64
Unique Fiscal Years in model_df_ord_wq: 5
Number of unique covariate patterns in X_mp_wq: 5

  n = 315 oysters (considering water quality data)
  MP coef = 71.9722, OR = 18077106489915233595349228584960.0000, p = 0.555216
  Direction (MP): Higher MP → more atrophy

  Water Quality Parameters:
    ph coef = 52.0477, OR = 40182696305244864774144.0000, p = 0.734045
    Direction (ph): Higher value → more atrophy
    no3 coef = -6.5963, OR = 0.0014, p = 0.364199
    Direction (no3): Higher value → less atrophy
    so coef = -12.7409, OR = 0.0000, p = 0.277697
    Direction (so): Higher value → less atroph

### STEP 9: GEE — Severe DTA ~ MP + Water Quality (Cluster-Robust Standard Errors)

Now, let's run a GEE model for 'severe DTA' that also incorporates water quality parameters, maintaining cluster-robust standard errors by year.

In [135]:
print(f"\n{'=' * 70}")
print("STEP 9: GEE — Severe DTA ~ MP + Water Quality")
print("=" * 70)

# Prepare data for GEE model with water quality
gee_df_wq = oy_dta_regional_wq[['severe', 'mp_regional_mean', 'Fiscal Year', 'ph', 'no3', 'to', 'so']].dropna().copy()
gee_df_wq = gee_df_wq.sort_values('Fiscal Year')

if gee_df_wq['Fiscal Year'].nunique() >= 3 and not gee_df_wq.empty:
    # Naive logistic with WQ
    naive_wq = smf.logit('severe ~ mp_regional_mean + ph + no3 + to + so', data=gee_df_wq).fit(disp=0)

    # GEE clustered by year with WQ
    gee_year_wq = gee.GEE.from_formula(
        'severe ~ mp_regional_mean + ph + no3 + to + so',
        groups='Fiscal Year', data=gee_df_wq,
        family=Binomial(), cov_struct=Exchangeable()
    ).fit()

    se_naive_wq = naive_wq.bse['mp_regional_mean']
    # Numerical instability in GEE often results in NaNs for SEs and p-values
    # especially with few clusters and many predictors.
    se_gee_wq   = gee_year_wq.bse['mp_regional_mean'] if 'mp_regional_mean' in gee_year_wq.bse else np.nan
    se_inflation_wq = se_gee_wq / se_naive_wq if not np.isnan(se_gee_wq) else np.nan

    print(f"\n{'Method':<40} {'Coef':>10} {'SE':>10} {'p-value':>10} {'OR':>10}")
    print("-" * 80)
    print(f"{'Naive logistic (MP + WQ, inflated n):':<40} {naive_wq.params['mp_regional_mean']:>10.4f}"
          f" {se_naive_wq:>10.4f} {naive_wq.pvalues['mp_regional_mean']:>10.6f}"
          f" {np.exp(naive_wq.params['mp_regional_mean']):>10.4f}")

    # Handle potential NaN for GEE results for printing
    gee_mp_coef = gee_year_wq.params['mp_regional_mean'] if 'mp_regional_mean' in gee_year_wq.params else np.nan
    gee_mp_p_value = gee_year_wq.pvalues['mp_regional_mean'] if 'mp_regional_mean' in gee_year_wq.pvalues else np.nan
    gee_mp_or = np.exp(gee_mp_coef) if not np.isnan(gee_mp_coef) else np.nan

    print(f"{'GEE (MP + WQ, clustered by year):':<40} {gee_mp_coef:>10.4f}"
          f" {se_gee_wq:>10.4f} {gee_mp_p_value:>10.6f}"
          f" {gee_mp_or:>10.4f}")

    print(f"\n  Clusters (effective n): {gee_df_wq['Fiscal Year'].nunique()}")
    print(f"  SE inflation: {se_inflation_wq:.1f}×" if not np.isnan(se_inflation_wq) else f"  SE inflation: {se_inflation_wq}×")

    print(f"\n  Water Quality Parameters (GEE):")
    for param in ['ph', 'no3', 'to', 'so']:
        if param in gee_year_wq.params:
            coef = gee_year_wq.params[param]
            p_val = gee_year_wq.pvalues[param] if param in gee_year_wq.pvalues else np.nan
            or_val = np.exp(coef)
            print(f"    {param} coef = {coef:.4f}, OR = {or_val:.4f}, p = {p_val:.6f}")
            print(f"    Direction ({param}): {'Higher value → more severe DTA' if coef > 0 else 'Higher value → less severe DTA'}")
else:
    print("  Insufficient data to run GEE with Water Quality.")

# Store for plotting later
gee_mp_coef_wq_val = gee_year_wq.params['mp_regional_mean'] if 'gee_year_wq' in locals() and 'mp_regional_mean' in gee_year_wq.params else np.nan
gee_mp_p_wq_val = gee_year_wq.pvalues['mp_regional_mean'] if 'gee_year_wq' in locals() and 'mp_regional_mean' in gee_year_wq.pvalues else np.nan


STEP 9: GEE — Severe DTA ~ MP + Water Quality

Method                                         Coef         SE    p-value         OR
--------------------------------------------------------------------------------
Naive logistic (MP + WQ, inflated n):       32.6422 147441.3873   0.999823 150078898846929.8438
GEE (MP + WQ, clustered by year):           32.6445     0.0006   0.000000 150429920852436.0938

  Clusters (effective n): 5
  SE inflation: 0.0×

  Water Quality Parameters (GEE):
    ph coef = -5.3023, OR = 0.0050, p = 0.000000
    Direction (ph): Higher value → less severe DTA
    no3 coef = -5.5564, OR = 0.0039, p = 0.000000
    Direction (no3): Higher value → less severe DTA
    to coef = -1.0599, OR = 0.3465, p = 0.000000
    Direction (to): Higher value → less severe DTA
    so coef = 1.8249, OR = 6.2023, p = 0.000000
    Direction (so): Higher value → more severe DTA


### Updated Model Comparison Plot

Let's update the model comparison plot to include the results from our new models incorporating water quality.

In [136]:
import os

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1)

# Original models
model_labels_orig = ['MP only\n(ordinal)', 'MP + Location\n(ordinal)', 'Severe DTA\n(GEE)']
model_coefs_orig = [mp_coef, mp_coef2, gee_year.params['mp_regional_mean']]
model_pvals_orig = [mp_p, mp_p2, gee_year.pvalues['mp_regional_mean']]

# New models with Water Quality
# Ensure these variables are defined and handled, as some might be NaN due to model failures or data limitations
mp_coef_wq_val_actual = mp_coef_wq if 'mp_coef_wq' in locals() else np.nan
mp_p_wq_val_actual = mp_p_wq if 'mp_p_wq' in locals() else np.nan
gee_mp_coef_wq_val_actual = gee_year_wq.params['mp_regional_mean'] if 'gee_year_wq' in locals() and 'mp_regional_mean' in gee_year_wq.params else np.nan
gee_mp_p_wq_val_actual = gee_year_wq.pvalues['mp_regional_mean'] if 'gee_year_wq' in locals() and 'mp_regional_mean' in gee_year_wq.pvalues else np.nan

model_labels_new = ['MP + WQ\n(ordinal)', 'Severe DTA + WQ\n(GEE)']
model_coefs_new = [mp_coef_wq_val_actual, gee_mp_coef_wq_val_actual]
model_pvals_new = [mp_p_wq_val_actual, gee_mp_p_wq_val_actual]

# Combine all model results
all_model_labels = model_labels_orig + model_labels_new
all_model_coefs = model_coefs_orig + model_coefs_new
all_model_pvals = model_pvals_orig + all_model_pvals # mp_p_wq_val is now `np.nan` from the last run for Ordinal Logistic. Use the updated list for the second index.

# Handle the Ordinal Logistic model (MP + WQ) which failed with nan values for coefficient and p-value.
# The `all_model_pvals` variable in the kernel state is `[0.11940382738348314, 0.5723835636138169, 0.38982841842975624, 0.6161398112758565, nan]`.
# The last value `nan` corresponds to 'Severe DTA + WQ (GEE)', which was wrongly assigned.
# The correct `p` value is 0.0 for `gee_mp_p_wq_val_actual`.
# The p-value for the failed ordinal model should reflect its failure (e.g., NaN). So we are using `mp_p_wq_val_actual` for the ordinal model and `gee_mp_p_wq_val_actual` for the GEE model.
all_model_pvals = model_pvals_orig + [mp_p_wq_val_actual, gee_mp_p_wq_val_actual]

m_cols = ['#f44336' if (isinstance(p, (float, np.float64)) and p < 0.05) else '#9E9E9E' for p in all_model_pvals]
bars = ax.bar(all_model_labels, all_model_coefs, color=m_cols, edgecolor='black', lw=0.8, width=0.55)
ax.axhline(0, color='black', lw=0.8)
for b, c, pv in zip(bars, all_model_coefs, all_model_pvals):
    if not np.isnan(c) and not np.isnan(pv):
        sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else 'ns'
        yp = c + 0.3 if c > 0 else c - 0.3
        ax.text(b.get_x()+b.get_width()/2, yp, f'{c:.2f}\n({sig})',
                ha='center', va='bottom' if c > 0 else 'top', fontsize=8)
    elif np.isnan(c) and np.isnan(pv):
        # Handle cases where both coefficient and p-value are NaN (e.g., model failed)
        yp = b.get_height() + 0.3 # Position text above the bar if coefficient is NaN, default to 0 if bar is not drawn
        if np.isnan(b.get_height()): yp = 0.3 # If bar height is NaN, position above x-axis
        ax.text(b.get_x()+b.get_width()/2, yp, 'NaN\n(Fail)',
                ha='center', va='bottom', fontsize=8, color='gray')

ax.set_ylabel('MP Coefficient')
ax.set_title('MP Effect Across Models (Including Water Quality)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

# Ensure the output directory exists
os.makedirs(OUT_DIR, exist_ok=True)

plt.savefig(f'{OUT_DIR}/oyster_dta_mp_wq_model_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
print(f"  Saved: oyster_dta_mp_wq_model_comparison.png")
plt.show()

  Saved: oyster_dta_mp_wq_model_comparison.png


In [137]:
import pandas as pd

water_quality_url = 'https://raw.githubusercontent.com/dhairya-p/bc3412-code/main/water_quality.csv'
water_quality_df = pd.read_csv(water_quality_url)

print(f"Water Quality: {len(water_quality_df):,} records")
display(water_quality_df.head())

Water Quality: 144 records


,year,month,ph,o2,no3,po4,si,fe,spco2,to,so
0,2004,1,8.122864,250.97887,2.589186,0.256944,4.639858,0.001242,31.468166,12.229954,34.301690
1,2004,2,8.121401,265.07758,3.207088,0.303460,4.855079,0.001215,31.515516,11.117142,34.319280
2,2004,3,8.121297,273.65164,2.250334,0.246508,4.086512,0.001130,31.332727,11.007979,34.261654
3,2004,4,8.114388,274.34710,1.113749,0.164480,3.379702,0.001128,31.846405,12.066129,34.038395
4,2004,5,8.072080,259.42400,0.297408,0.096970,3.437382,0.001299,35.716300,15.774302,33.856884


In [138]:
import os

# Create the output directory if it doesn't exist
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Created output directory: {OUT_DIR}")

Created output directory: /mnt/user-data/outputs
